# RAG Prototype 1 — Ingestion

Imports → configuration → data acquisition → document extraction → ingestion run. Chunking is the next notebook.

## 1. Imports

In [1]:
import json
import logging
import re
import time
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd
import pymupdf
import requests
from docx import Document
from openpyxl import load_workbook


## 2. Configuration & Logging

`data/` is now split into `data/raw/` (exactly what was downloaded or handed to you — never modified) and `data/processed/` (what ingestion produces — safe to delete and regenerate at any time). Every tunable number used later (thresholds for "is this column an id?", "is this line a repeated header?", etc.) lives here too, instead of being buried inside the function that uses it.

In [2]:
@dataclass
class Config:
    base_dir: Path = field(default_factory=Path.cwd)
    data_subdir: str = "data"
    raw_subdir: str = "raw"                # data/raw       — untouched source files
    processed_subdir: str = "processed"    # data/processed — ingestion output (regenerable)
    output_subdir: str = "outputs"
    log_subdir: str = "logs"

    supported_extensions: tuple = (".pdf", ".docx", ".xlsx")
    download_timeout_s: int = 30
    download_retries: int = 3
    download_retry_backoff_s: float = 2.0

    short_page_word_threshold: int = 20

    # A line that repeats across at least this fraction of a PDF's pages is
    # treated as a running header/footer and stripped.
    pdf_boilerplate_min_repeat_ratio: float = 0.4

    # An xlsx column whose non-empty values average fewer words than this
    # is treated as an identifier/category rather than free text.
    xlsx_semantic_min_avg_words: float = 3.0
    xlsx_id_like_names: frozenset = frozenset({
        "id", "row_id", "index", "rank", "ranking", "serial", "sr_no", "s_no",
    })

    @property
    def data_dir(self) -> Path:
        return self.base_dir / self.data_subdir

    @property
    def raw_dir(self) -> Path:
        return self.data_dir / self.raw_subdir

    @property
    def processed_dir(self) -> Path:
        return self.data_dir / self.processed_subdir

    @property
    def output_dir(self) -> Path:
        return self.base_dir / self.output_subdir

    @property
    def log_dir(self) -> Path:
        return self.base_dir / self.log_subdir

    def ensure_dirs(self):
        for directory in (self.raw_dir, self.processed_dir, self.output_dir, self.log_dir):
            directory.mkdir(exist_ok=True, parents=True)


CONFIG = Config()
CONFIG.ensure_dirs()

print(f"Raw directory:       {CONFIG.raw_dir}")
print(f"Processed directory: {CONFIG.processed_dir}")
print(f"Output directory:    {CONFIG.output_dir}")
print(f"Log directory:       {CONFIG.log_dir}")


Raw directory:       c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\data\raw
Processed directory: c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\data\processed
Output directory:    c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\outputs
Log directory:       c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\logs


**One-time manual step:** your existing files (`Attention_Is_All_You_Need.docx`, `Foundation-LLMs.pdf`, `rag.xlsx`, the RAG paper `.docx`) are sitting directly in `data/` from before this change. Move them into `data/raw/` — I can't touch your local filesystem from here, so this one's on you. Everything from this point on reads from `raw_dir` and writes to `processed_dir`.

In [3]:
logger = logging.getLogger("rag_ingestion")
logger.setLevel(logging.INFO)
logger.handlers.clear()  # avoid duplicate handlers if this cell is re-run

_formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

_console_handler = logging.StreamHandler()
_console_handler.setFormatter(_formatter)
logger.addHandler(_console_handler)

_file_handler = logging.FileHandler(CONFIG.log_dir / "ingestion.log")
_file_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)

logger.info("Logger initialized.")


2026-08-26 22:59:56,110 | INFO | Logger initialized.


## 3. Data Acquisition

Downloads now land in `raw_dir`, retrying on transient failures and skipping files that are already present.

In [4]:
'''DATA_SOURCES = [
    {
        "title": "Attention Is All You Need",
        "url": "https://arxiv.org/pdf/1706.03762",
    },
    {
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "url": "https://arxiv.org/pdf/2005.11401",
    },
]

DATA_SOURCES'''


'DATA_SOURCES = [\n    {\n        "title": "Attention Is All You Need",\n        "url": "https://arxiv.org/pdf/1706.03762",\n    },\n    {\n        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",\n        "url": "https://arxiv.org/pdf/2005.11401",\n    },\n]\n\nDATA_SOURCES'

In [5]:
'''def download_sources(sources: list[dict], config: Config = CONFIG) -> list[Path]:
    """Download each source PDF into config.raw_dir, skipping files that already exist."""
    downloaded = []

    for source in sources:
        filename = source["title"].replace(" ", "_") + ".pdf"
        file_path = config.raw_dir / filename

        if file_path.exists():
            logger.info(f"Already have {filename}, skipping download.")
            downloaded.append(file_path)
            continue

        for attempt in range(1, config.download_retries + 1):
            try:
                response = requests.get(source["url"], timeout=config.download_timeout_s)
                response.raise_for_status()

                file_path.write_bytes(response.content)
                logger.info(f"Downloaded: {filename}")
                downloaded.append(file_path)
                break

            except requests.RequestException as exc:
                logger.warning(
                    f"Attempt {attempt}/{config.download_retries} failed for "
                    f"{source['title']!r}: {exc}"
                )
                if attempt < config.download_retries:
                    time.sleep(config.download_retry_backoff_s * attempt)
                else:
                    logger.error(f"Giving up on {source['title']!r} after {attempt} attempts.")

    return downloaded


download_sources(DATA_SOURCES)'''


'def download_sources(sources: list[dict], config: Config = CONFIG) -> list[Path]:\n    """Download each source PDF into config.raw_dir, skipping files that already exist."""\n    downloaded = []\n\n    for source in sources:\n        filename = source["title"].replace(" ", "_") + ".pdf"\n        file_path = config.raw_dir / filename\n\n        if file_path.exists():\n            logger.info(f"Already have {filename}, skipping download.")\n            downloaded.append(file_path)\n            continue\n\n        for attempt in range(1, config.download_retries + 1):\n            try:\n                response = requests.get(source["url"], timeout=config.download_timeout_s)\n                response.raise_for_status()\n\n                file_path.write_bytes(response.content)\n                logger.info(f"Downloaded: {filename}")\n                downloaded.append(file_path)\n                break\n\n            except requests.RequestException as exc:\n                logger.warning(\n

## 4. Data Inventory

In [6]:
def list_data_files(config: Config = CONFIG) -> list[Path]:
    files = sorted(config.raw_dir.iterdir())

    logger.info(f"Files in raw directory: {len(files)}")
    for file in files:
        size_kb = file.stat().st_size / 1024
        logger.info(f"- {file.name} | {file.suffix or 'no ext'} | {size_kb:.2f} KB")

    return files


data_files = list_data_files()


2026-08-26 22:59:56,164 | INFO | Files in raw directory: 4
2026-08-26 22:59:56,165 | INFO | - Attention_Is_All_You_Need.docx | .docx | 330.47 KB
2026-08-26 22:59:56,167 | INFO | - Foundation-LLMs.pdf | .pdf | 2656.22 KB
2026-08-26 22:59:56,168 | INFO | - rag.xlsx | .xlsx | 898.20 KB
2026-08-26 22:59:56,169 | INFO | - Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.docx | .docx | 245.31 KB


## 5. Document Extraction

### 5.1 Shared helpers

`clean_text()` and `compute_stats()` run for every record regardless of source type, so every record ends up with the same cleaned `text` and the same `char_count` / `word_count` / `line_count` fields.

In [7]:
def clean_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def compute_stats(text: str) -> dict:
    return {
        "char_count": len(text),
        "word_count": len(text.split()),
        "line_count": len(text.splitlines()),
    }


def make_record(document: str, source_type: str, location: str, text: str, metadata: dict | None = None) -> dict:
    cleaned = clean_text(text)
    record = {
        "document": document,
        "source_type": source_type,
        "location": location,
        "text": cleaned,
        "metadata": metadata or {},
    }
    record.update(compute_stats(cleaned))
    return record


### 5.2 PDF — with running header/footer removal

This formalizes the line-frequency check you were doing manually with `Counter`: a line that shows up on a large fraction of a document's pages (a running header, a page footer, a copyright notice) carries no content and just adds noise to every chunk. `find_boilerplate_lines()` finds those lines per-document, and `extract_pdf` strips them before the text is stored.

In [8]:
def find_boilerplate_lines(page_texts: list[str], config: Config = CONFIG) -> set[str]:
    """Lines that repeat across a large fraction of a document's pages — treated as
    running headers/footers rather than content."""
    if len(page_texts) < 3:
        return set()  # too few pages for repetition to mean anything

    line_counts = Counter()
    for text in page_texts:
        unique_lines_on_page = {line.strip() for line in text.splitlines() if line.strip()}
        line_counts.update(unique_lines_on_page)

    threshold = max(2, int(len(page_texts) * config.pdf_boilerplate_min_repeat_ratio))
    return {line for line, count in line_counts.items() if count >= threshold}


def remove_lines(text: str, lines_to_remove: set[str]) -> str:
    if not lines_to_remove:
        return text
    kept = [line for line in text.splitlines() if line.strip() not in lines_to_remove]
    return "\n".join(kept)


In [9]:
def extract_pdf(file_path: Path, config: Config = CONFIG) -> list[dict]:
    with pymupdf.open(file_path) as doc:
        page_texts = [page.get_text() for page in doc]

    boilerplate = find_boilerplate_lines(page_texts, config)
    if boilerplate:
        logger.info(f"{file_path.name}: stripping {len(boilerplate)} repeated header/footer line(s)")

    documents = []
    for page_number, text in enumerate(page_texts, start=1):
        text = remove_lines(text, boilerplate)
        documents.append(
            make_record(
                document=file_path.name,
                source_type="pdf",
                location=f"page_{page_number}",
                text=text,
            )
        )

    return documents


### 5.3 DOCX

Same as before, plus the paragraph's Word style (`Heading 1`, `Normal`, etc.) is kept in `metadata` — cheap to capture now, and useful later for telling headings apart from body text when chunking.

In [10]:
def extract_docx(file_path: Path) -> list[dict]:
    documents = []

    doc = Document(file_path)

    for index, paragraph in enumerate(doc.paragraphs, start=1):
        text = paragraph.text.strip()

        if text:
            documents.append(
                make_record(
                    document=file_path.name,
                    source_type="docx",
                    location=f"paragraph_{index}",
                    text=text,
                    metadata={"style": paragraph.style.name},
                )
            )

    return documents


### 5.4 XLSX — column classification (simplified)

Your instinct here was right, and worth keeping: not every column in a spreadsheet is meant to be embedded as text. An `id` column or a short category column just adds noise if it's mashed into the same string as the real content. The scoring system you built (name signal + unique ratio + repetition ratio + missing ratio, each weighted) was measuring the right thing but with more machinery than a prototype needs.

The simplified version uses two signals only:
- **name** — does the header look like `id`, `index`, `rank`, etc.?
- **average word count** — free text (`question`, `answer`) reads as multiple words per cell; identifiers and short categories read as one or two.

`semantic` columns are joined into `text` (what actually gets embedded later). `metadata` columns are attached to `metadata` on the record instead of thrown away — still there if you need to filter or trace a chunk back to its row.

In [11]:
def classify_xlsx_columns(headers: list[str], data_rows: list[tuple], config: Config = CONFIG) -> dict[str, str]:
    columns = {header: [] for header in headers}
    for row in data_rows:
        for header, value in zip(headers, row):
            columns[header].append(value)

    classification = {}
    for header, values in columns.items():
        non_empty = [str(v).strip() for v in values if v is not None and str(v).strip()]
        avg_word_count = (
            sum(len(v.split()) for v in non_empty) / len(non_empty)
            if non_empty else 0
        )

        looks_like_id = header.strip().lower() in config.xlsx_id_like_names
        looks_short = avg_word_count < config.xlsx_semantic_min_avg_words

        classification[header] = "metadata" if (looks_like_id or looks_short) else "semantic"

    return classification


Preview the classification before running full ingestion — worth a glance so a real free-text column doesn't quietly get misclassified as metadata.

In [12]:
def read_xlsx_rows(file_path: Path) -> tuple[list[str], list[tuple]]:
    workbook = load_workbook(file_path, read_only=True, data_only=True)
    preview_rows = []

    for sheet in workbook.worksheets:
        rows = list(sheet.iter_rows(values_only=True))
        if not rows:
            continue

        headers = [
            str(value).strip() if value is not None else f"column_{i}"
            for i, value in enumerate(rows[0])
        ]
        preview_rows.append((sheet.title, headers, rows[1:]))

    workbook.close()
    return preview_rows


column_preview = []
for file_path in CONFIG.raw_dir.glob("*.xlsx"):
    for sheet_title, headers, data_rows in read_xlsx_rows(file_path):
        roles = classify_xlsx_columns(headers, data_rows)
        for header, role in roles.items():
            column_preview.append({
                "file": file_path.name,
                "sheet": sheet_title,
                "column": header,
                "role": role,
            })

pd.DataFrame(column_preview)


,file,sheet,column,role
0,rag.xlsx,Sheet1,question,semantic
1,rag.xlsx,Sheet1,answer,semantic
2,rag.xlsx,Sheet1,relevant_passage_ids,semantic
3,rag.xlsx,Sheet1,id,metadata


In [13]:
def extract_xlsx(file_path: Path, config: Config = CONFIG) -> list[dict]:
    documents = []

    for sheet_title, headers, data_rows in read_xlsx_rows(file_path):
        column_roles = classify_xlsx_columns(headers, data_rows, config)
        semantic_headers = [h for h in headers if column_roles[h] == "semantic"]
        metadata_headers = [h for h in headers if column_roles[h] == "metadata"]

        logger.info(
            f"{file_path.name} [{sheet_title}]: semantic={semantic_headers} metadata={metadata_headers}"
        )

        for row_number, row in enumerate(data_rows, start=2):  # row 1 was the header
            row_dict = dict(zip(headers, row))

            semantic_values = [
                str(row_dict[h]).strip()
                for h in semantic_headers
                if row_dict.get(h) is not None and str(row_dict[h]).strip()
            ]
            if not semantic_values:
                continue

            text = " | ".join(semantic_values)

            row_metadata = {"sheet": sheet_title, "row": row_number}
            for h in metadata_headers:
                if row_dict.get(h) is not None:
                    row_metadata[h] = row_dict[h]

            documents.append(
                make_record(
                    document=file_path.name,
                    source_type="xlsx",
                    location=f"{sheet_title}!row_{row_number}",
                    text=text,
                    metadata=row_metadata,
                )
            )

    return documents


In [14]:
EXTRACTORS = {
    ".pdf": extract_pdf,
    ".docx": extract_docx,
    ".xlsx": extract_xlsx,
}


def ingest_file(file_path: Path) -> list[dict]:
    suffix = file_path.suffix.lower()

    extractor = EXTRACTORS.get(suffix)
    if extractor is None:
        raise ValueError(f"Unsupported file format: {suffix}")

    return extractor(file_path)


## 6. Run Ingestion

Reads from `raw_dir`, wraps each file in its own `try/except`, and saves the final record list to `processed_dir` as JSON — so the chunking notebook can just load that file instead of re-running extraction.

In [15]:
def run_ingestion(config: Config = CONFIG) -> list[dict]:
    all_documents = []

    for file_path in sorted(config.raw_dir.iterdir()):
        if file_path.suffix.lower() not in config.supported_extensions:
            continue

        try:
            extracted = ingest_file(file_path)
            all_documents.extend(extracted)
            logger.info(f"{file_path.name}: {len(extracted)} records")

        except Exception:
            logger.exception(f"Failed to ingest {file_path.name}, skipping.")

    logger.info(f"Total records: {len(all_documents)}")
    return all_documents


def save_documents(documents: list[dict], config: Config = CONFIG, filename: str = "ingested_records.json") -> Path:
    out_path = config.processed_dir / filename
    out_path.write_text(json.dumps(documents, indent=2, ensure_ascii=False), encoding="utf-8")
    logger.info(f"Saved {len(documents)} records to {out_path}")
    return out_path


documents = run_ingestion()
save_documents(documents)


2026-08-26 22:59:57,755 | INFO | Attention_Is_All_You_Need.docx: 298 records
2026-08-26 22:59:58,999 | INFO | Foundation-LLMs.pdf: 277 records
2026-08-26 22:59:59,742 | INFO | rag.xlsx [Sheet1]: semantic=['question', 'answer', 'relevant_passage_ids'] metadata=['id']
2026-08-26 23:00:00,005 | INFO | rag.xlsx: 4719 records
2026-08-26 23:00:00,557 | INFO | Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.docx: 211 records
2026-08-26 23:00:00,558 | INFO | Total records: 5505
2026-08-26 23:00:00,746 | INFO | Saved 5505 records to c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\data\processed\ingested_records.json


WindowsPath('c:/Users/Swapn/OneDrive/Documents/GitHub/Surfprice/data/processed/ingested_records.json')

### Sanity checks

In [16]:
empty_docs = [doc for doc in documents if not doc["text"]]

print(f"Total records: {len(documents)}")
print(f"Empty records: {len(empty_docs)}")
print(f"Non-empty records: {len(documents) - len(empty_docs)}")


Total records: 5505
Empty records: 0
Non-empty records: 5505


In [17]:
short_docs = [
    doc for doc in documents
    if doc["word_count"] < CONFIG.short_page_word_threshold
]

print(f"Suspiciously short records (< {CONFIG.short_page_word_threshold} words): {len(short_docs)}")

for doc in short_docs[:10]:
    print(doc["document"], "|", doc["location"], "| words:", doc["word_count"])


Suspiciously short records (< 20 words): 619
Attention_Is_All_You_Need.docx | paragraph_3 | words: 5
Attention_Is_All_You_Need.docx | paragraph_10 | words: 5
Attention_Is_All_You_Need.docx | paragraph_11 | words: 5
Attention_Is_All_You_Need.docx | paragraph_12 | words: 5
Attention_Is_All_You_Need.docx | paragraph_13 | words: 5
Attention_Is_All_You_Need.docx | paragraph_17 | words: 5
Attention_Is_All_You_Need.docx | paragraph_18 | words: 8
Attention_Is_All_You_Need.docx | paragraph_19 | words: 2
Attention_Is_All_You_Need.docx | paragraph_20 | words: 2
Attention_Is_All_You_Need.docx | paragraph_21 | words: 1


Ingestion's done — `documents` (and the saved copy in `data/processed/`) is ready for chunking.

In [18]:
def check_structure_awareness(documents: list[dict]) -> dict:
    """Do these records carry structural signals, and are they already
    grouped into sections? (Signals: maybe. Grouping: no, not yet.)"""
    report = {}

    for source_type in {d["source_type"] for d in documents}:
        subset = [d for d in documents if d["source_type"] == source_type]

        metadata_keys = set()
        for d in subset:
            metadata_keys.update(d["metadata"].keys())

        has_heading_signal = any(
            "heading" in str(d["metadata"].get("style", "")).lower()
            for d in subset
        )

        # A record would only be "grouped" if something links it to a parent
        # section — e.g. a "section_id" field. Nothing produces that yet.
        has_section_grouping = any(
            "section" in d or "section" in d["metadata"]
            for d in subset
        )

        report[source_type] = {
            "record_count": len(subset),
            "metadata_keys_present": sorted(metadata_keys),
            "has_heading_style_signal": has_heading_signal,
            "has_explicit_section_grouping": has_section_grouping,
        }

    return report


for source_type, info in check_structure_awareness(documents).items():
    print(f"\n{source_type.upper()}")
    for key, value in info.items():
        print(f"  {key}: {value}")


DOCX
  record_count: 509
  metadata_keys_present: ['style']
  has_heading_style_signal: True
  has_explicit_section_grouping: False

XLSX
  record_count: 4719
  metadata_keys_present: ['id', 'row', 'sheet']
  has_heading_style_signal: False
  has_explicit_section_grouping: False

PDF
  record_count: 277
  metadata_keys_present: []
  has_heading_style_signal: False
  has_explicit_section_grouping: False


In [19]:
from dataclasses import dataclass, field
from typing import Any


@dataclass
class StructuralUnit:
    source_id: str
    source_type: str

    title: str | None
    level: int | None

    text: str

    metadata: dict[str, Any] = field(default_factory=dict)

In [20]:
StructuralUnit(
    source_id="manual_001",
    source_type="docx",
    title="Authentication",
    level=1,
    text="Authentication allows users to...",
    metadata={
        "section_path": ["Authentication"]
    }
)

StructuralUnit(source_id='manual_001', source_type='docx', title='Authentication', level=1, text='Authentication allows users to...', metadata={'section_path': ['Authentication']})

In [21]:
def is_heading(record: dict) -> bool:
    style = str(record["metadata"].get("style", "")).lower()

    return style.startswith("heading")

In [22]:
def heading_level(record: dict) -> int | None:
    style = str(record["metadata"].get("style", "")).lower()

    if style.startswith("heading"):
        try:
            return int(style.replace("heading", "").strip())
        except ValueError:
            return None

    return None

In [23]:
def build_docx_sections(records: list[dict]) -> list[StructuralUnit]:
    sections = []

    current_title = None
    current_level = None
    current_text = []

    section_path = []

    for record in records:

        text = record["text"].strip()

        if not text:
            continue

        if is_heading(record):

            if current_text:
                sections.append(
                    StructuralUnit(
                        source_id=record["source_id"],
                        source_type="docx",
                        title=current_title,
                        level=current_level,
                        text="\n".join(current_text),
                        metadata={
                            "section_path": section_path.copy()
                        },
                    )
                )

            level = heading_level(record)

            current_title = text
            current_level = level

            section_path = section_path[: max(level - 1, 0)]
            section_path.append(text)

            current_text = []

        else:
            current_text.append(text)

    if current_text:
        sections.append(
            StructuralUnit(
                source_id=records[0]["source_id"],
                source_type="docx",
                title=current_title,
                level=current_level,
                text="\n".join(current_text),
                metadata={
                    "section_path": section_path.copy()
                },
            )
        )

    return sections

In [24]:
from collections import Counter

print("Total records:", len(documents))

for source_type, count in Counter(
    d["source_type"] for d in documents
).items():
    print(f"{source_type}: {count}")

print("\nExample record:")
print(documents[0])

Total records: 5505
docx: 509
pdf: 277
xlsx: 4719

Example record:
{'document': 'Attention_Is_All_You_Need.docx', 'source_type': 'docx', 'location': 'paragraph_1', 'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.', 'metadata': {'style': 'Normal'}, 'char_count': 173, 'word_count': 26, 'line_count': 1}


In [25]:
from collections import Counter

docx_records = [
    d for d in documents
    if d["source_type"] == "docx"
]

style_counts = Counter(
    d["metadata"].get("style", "UNKNOWN")
    for d in docx_records
)

print("DOCX records:", len(docx_records))
print("\nStyles found:")

for style, count in style_counts.most_common():
    print(f"{style}: {count}")

DOCX records: 509

Styles found:
Normal: 354
Body Text: 87
List Paragraph: 43
Heading 2: 15
Heading 1: 10


In [26]:
def is_docx_heading(record):
    style = record.get("metadata", {}).get("style", "")
    return style in {"Heading 1", "Heading 2"}

In [27]:
def get_docx_heading_level(record):
    style = record.get("metadata", {}).get("style", "")

    if style == "Heading 1":
        return 1

    if style == "Heading 2":
        return 2

    return None

In [28]:
def build_docx_sections(records):
    sections = []

    current_title = None
    current_level = None
    current_text = []

    section_path = []

    for record in records:

        text = record.get("text", "").strip()

        if not text:
            continue

        if is_docx_heading(record):

            # Save previous section
            if current_text:
                sections.append({
                    "document": record["document"],
                    "source_type": "docx",
                    "title": current_title,
                    "level": current_level,
                    "text": "\n".join(current_text),
                    "metadata": {
                        "section_path": section_path.copy()
                    }
                })

            level = get_docx_heading_level(record)

            current_title = text
            current_level = level

            # Maintain hierarchy
            section_path = section_path[:level - 1]
            section_path.append(text)

            current_text = []

        else:
            current_text.append(text)

    # Save final section
    if current_text:
        sections.append({
            "document": records[0]["document"],
            "source_type": "docx",
            "title": current_title,
            "level": current_level,
            "text": "\n".join(current_text),
            "metadata": {
                "section_path": section_path.copy()
            }
        })

    return sections

In [29]:
heading_docs = {
    d["document"]
    for d in docx_records
    if is_docx_heading(d)
}

print("Documents containing headings:")

for doc in heading_docs:
    print(doc)

Documents containing headings:
Attention_Is_All_You_Need.docx


In [30]:
test_document = next(iter(heading_docs))

test_records = [
    d for d in docx_records
    if d["document"] == test_document
]

sections = build_docx_sections(test_records)

print("Document:", test_document)
print("Sections:", len(sections))

Document: Attention_Is_All_You_Need.docx
Sections: 25


In [31]:
for i, section in enumerate(sections[:10]):

    print("=" * 80)
    print("SECTION:", i)
    print("TITLE:", section["title"])
    print("LEVEL:", section["level"])
    print("PATH:", section["metadata"]["section_path"])
    print("TEXT PREVIEW:", section["text"][:500])

SECTION: 0
TITLE: None
LEVEL: None
PATH: []
TEXT PREVIEW: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.
Attention Is All You Need
Ashish Vaswani∗ Google Brain avaswani@google.com
Noam Shazeer∗ Google Brain noam@google.com
Niki Parmar∗ Google Research nikip@google.com
Jakob Uszkoreit∗ Google Research usz@google.com
Llion Jones∗ Google Research llion@google.com
Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu
Łukasz 
SECTION: 1
TITLE: Abstract
LEVEL: 1
PATH: ['Abstract']
TEXT PREVIEW: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and

In [32]:
section_stats = []

for document in {
    d["document"] for d in docx_records
}:

    records = [
        d for d in docx_records
        if d["document"] == document
    ]

    sections = build_docx_sections(records)

    section_stats.append({
        "document": document,
        "records": len(records),
        "sections": len(sections),
        "heading_sections": sum(
            s["title"] is not None
            for s in sections
        )
    })

for item in section_stats:
    print(item)

{'document': 'Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.docx', 'records': 211, 'sections': 1, 'heading_sections': 0}
{'document': 'Attention_Is_All_You_Need.docx', 'records': 298, 'sections': 25, 'heading_sections': 24}


In [33]:
rag_records = [
    d for d in docx_records
    if d["document"] == "Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.docx"
]

rag_style_counts = Counter(
    d.get("metadata", {}).get("style", "UNKNOWN")
    for d in rag_records
)

print("RAG document styles:")

for style, count in rag_style_counts.most_common():
    print(f"{style}: {count}")

RAG document styles:
Normal: 211


In [34]:
for record in rag_records[:30]:
    print(
        f"{record['location']} | "
        f"{record['metadata'].get('style')} | "
        f"{record['text'][:120]}"
    )

paragraph_2 | Normal | Retrieval-Augmented Generation for
paragraph_3 | Normal | Knowledge-Intensive NLP Tasks
paragraph_4 | Normal | Patrick Lewis†‡, Ethan Perez⋆,
paragraph_5 | Normal | seq2seq baseline.
paragraph_6 | Normal | 1 Introduction
paragraph_7 | Normal | Pre-trained neural language models have been shown to learn a substantial amount of in-depth knowl-edge from data [47].T
paragraph_10 | Normal | Question Generation
paragraph_11 | Normal | Figure 1: Overview of our approach.We combine a pre-trained retriever (Query Encoder + Document Index) with a pre-traine
paragraph_12 | Normal | but have only explored open-domain extractive question answering.Here, we bring hybrid parametric and non-parametric mem
paragraph_13 | Normal | We endow pre-trained, parametric-memory generation models with a non-parametric memory through a general-purpose fine-tu
paragraph_14 | Normal | There has been extensive previous work proposing architectures to enrich systems with non-parametric memory w

In [35]:
import re


def detect_numbered_heading(text):
    text = text.strip()

    pattern = r"^\d+(?:\.\d+)*\s+.+$"

    return bool(re.match(pattern, text))

In [36]:
for record in rag_records:
    text = record["text"].strip()

    if detect_numbered_heading(text):
        print(record["location"], "→", text)

paragraph_6 → 1 Introduction
paragraph_16 → 2 Methods
paragraph_24 → 2.1 Models
paragraph_32 → 2.2 Retriever:DPR
paragraph_35 → 2.3 Generator:BART
paragraph_37 → 2.4 Training
paragraph_43 → 2.5 Decoding
paragraph_47 → 3 Experiments
paragraph_49 → 3.1 Open-domain Question Answering
paragraph_51 → 3.2 Abstractive Question Answering
paragraph_57 → 3.3 Jeopardy Question Generation
paragraph_60 → 3.4 Fact Verification
paragraph_62 → 4 Results
paragraph_63 → 4.1 Open-domain Question Answering
paragraph_77 → 4.2 Abstractive Question Answering
paragraph_79 → 4.3 Jeopardy Question Generation
paragraph_82 → 4.4 Fact Verification
paragraph_93 → 6 Discussion
paragraph_236 → 2
3


In [37]:
def detect_docx_structure(record):
    style = record.get("metadata", {}).get("style", "")

    if style == "Heading 1":
        return 1

    if style == "Heading 2":
        return 2

    text = record.get("text", "").strip()

    if detect_numbered_heading(text):
        return text.count(".") + 1

    return None

In [38]:
import re


def detect_docx_structure(record):
    style = record.get("metadata", {}).get("style", "")

    # Primary signal: DOCX heading styles
    if style == "Heading 1":
        return 1

    if style == "Heading 2":
        return 2

    # Fallback signal: numbered headings in text
    text = record.get("text", "").strip()

    match = re.match(r"^(\d+(?:\.\d+)*)\s+.+$", text)

    if match:
        number = match.group(1)
        return number.count(".") + 1

    return None

In [39]:
def build_docx_sections(records):
    sections = []

    current_title = None
    current_level = None
    current_text = []

    section_path = []

    for record in records:

        text = record.get("text", "").strip()

        if not text:
            continue

        level = detect_docx_structure(record)

        if level is not None:

            # Save previous section
            if current_text:
                sections.append({
                    "document": record["document"],
                    "source_type": "docx",
                    "title": current_title,
                    "level": current_level,
                    "text": "\n".join(current_text),
                    "metadata": {
                        "section_path": section_path.copy()
                    }
                })

            current_title = text
            current_level = level

            # Maintain hierarchy
            section_path = section_path[:level - 1]
            section_path.append(text)

            current_text = []

        else:
            current_text.append(text)

    # Save final section
    if current_text:
        sections.append({
            "document": records[0]["document"],
            "source_type": "docx",
            "title": current_title,
            "level": current_level,
            "text": "\n".join(current_text),
            "metadata": {
                "section_path": section_path.copy()
            }
        })

    return sections

In [40]:
rag_sections = build_docx_sections(rag_records)

print("Sections:", len(rag_sections))

for i, section in enumerate(rag_sections):
    print("=" * 80)
    print("SECTION:", i)
    print("TITLE:", section["title"])
    print("LEVEL:", section["level"])
    print("PATH:", section["metadata"]["section_path"])
    print("TEXT:", section["text"][:300])

Sections: 19
SECTION: 0
TITLE: None
LEVEL: None
PATH: []
TEXT: Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
seq2seq baseline.
SECTION: 1
TITLE: 1 Introduction
LEVEL: 1
PATH: ['1 Introduction']
TEXT: Pre-trained neural language models have been shown to learn a substantial amount of in-depth knowl-edge from data [47].They can do so without any access to an external memory, as a parameterized implicit knowledge base [51, 52].While this development is exciting, such models do have down-sides:They 
SECTION: 2
TITLE: 2 Methods
LEVEL: 1
PATH: ['2 Methods']
TEXT: We explore RAG models, which use the input sequence x to retrieve text documents z and use them asadditionalcontextwhengeneratingthetargetsequencey.AsshowninFigure1,ourmodels leverage two components:(i) a retriever pη(z|x) with parameters ηthat returns (top-K truncated) distributions over text passa
SECTION: 3
TITLE: 2.1 Models
LEVEL: 2
PATH: ['2 Methods', '2.1 Models']
TEXT: RAG-Seq

In [41]:
from dataclasses import dataclass, field
from typing import Any


@dataclass
class StructuralUnit:
    document: str
    source_type: str

    title: str | None
    level: int | None

    text: str

    metadata: dict[str, Any] = field(default_factory=dict)

In [42]:
def sections_to_structural_units(sections):
    return [
        StructuralUnit(
            document=section["document"],
            source_type=section["source_type"],
            title=section["title"],
            level=section["level"],
            text=section["text"],
            metadata=section["metadata"],
        )
        for section in sections
    ]

In [43]:
docx_units = sections_to_structural_units(rag_sections)

print("Structural units:", len(docx_units))
print()
print(docx_units[1])

Structural units: 19

StructuralUnit(document='Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.docx', source_type='docx', title='1 Introduction', level=1, text='Pre-trained neural language models have been shown to learn a substantial amount of in-depth knowl-edge from data [47].They can do so without any access to an external memory, as a parameterized implicit knowledge base [51, 52].While this development is exciting, such models do have down-sides:They cannot easily expand or revise their memory, can’t straightforwardly provide insight into their predictions, and may produce “hallucinations” [38].Hybrid models that combine parametric memory with non-parametric (i.e., retrieval-based) memories [20, 26, 48] can address some of these issues because knowledge can be directly revised and expanded, and accessed knowledge can be inspectedandinterpreted.REALM[20]andORQA[31],tworecentlyintroducedmodelsthat combine masked language models [8] with a differentiable retriever, 

In [44]:
@dataclass
class Chunk:
    chunk_id: str

    document: str
    source_type: str

    text: str

    metadata: dict[str, Any] = field(default_factory=dict)

In [45]:
print("Units:", len(docx_units))

for unit in docx_units[:5]:
    print("-" * 80)
    print("Document:", unit.document)
    print("Title:", unit.title)
    print("Level:", unit.level)
    print("Path:", unit.metadata.get("section_path"))
    print("Characters:", len(unit.text))

Units: 19
--------------------------------------------------------------------------------
Document: Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.docx
Title: None
Level: None
Path: []
Characters: 113
--------------------------------------------------------------------------------
Document: Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.docx
Title: 1 Introduction
Level: 1
Path: ['1 Introduction']
Characters: 4072
--------------------------------------------------------------------------------
Document: Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.docx
Title: 2 Methods
Level: 1
Path: ['2 Methods']
Characters: 1240
--------------------------------------------------------------------------------
Document: Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.docx
Title: 2.1 Models
Level: 2
Path: ['2 Methods', '2.1 Models']
Characters: 1219
--------------------------------------------------------------------------------
Do

In [46]:
def chunk_structural_unit(
    unit,
    chunk_size=1500,
    overlap=200
):
    text = unit.text.strip()

    if not text:
        return []

    if len(text) <= chunk_size:
        return [{
            "document": unit.document,
            "source_type": unit.source_type,
            "text": text,
            "metadata": {
                **unit.metadata,
                "section_title": unit.title,
                "section_level": unit.level,
                "chunk_index": 0,
            }
        }]

    chunks = []

    start = 0
    chunk_index = 0

    while start < len(text):

        end = min(start + chunk_size, len(text))

        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "document": unit.document,
                "source_type": unit.source_type,
                "text": chunk_text,
                "metadata": {
                    **unit.metadata,
                    "section_title": unit.title,
                    "section_level": unit.level,
                    "chunk_index": chunk_index,
                }
            })

            chunk_index += 1

        if end >= len(text):
            break

        start = end - overlap

    return chunks

In [47]:
intro_unit = next(
    unit for unit in docx_units
    if unit.title == "1 Introduction"
)

intro_chunks = chunk_structural_unit(intro_unit)

print("Original characters:", len(intro_unit.text))
print("Chunks:", len(intro_chunks))

for i, chunk in enumerate(intro_chunks):
    print("=" * 80)
    print("Chunk:", i)
    print("Characters:", len(chunk["text"]))
    print("Section:", chunk["metadata"]["section_title"])
    print("Path:", chunk["metadata"]["section_path"])
    print("Preview:", chunk["text"][:200])

Original characters: 4072
Chunks: 3
Chunk: 0
Characters: 1500
Section: 1 Introduction
Path: ['1 Introduction']
Preview: Pre-trained neural language models have been shown to learn a substantial amount of in-depth knowl-edge from data [47].They can do so without any access to an external memory, as a parameterized impli
Chunk: 1
Characters: 1500
Section: 1 Introduction
Path: ['1 Introduction']
Preview: e question answering.Here, we bring hybrid parametric and non-parametric memory to the “workhorse of NLP,” i.e.sequence-to-sequence (seq2seq) models.
We endow pre-trained, parametric-memory generation
Chunk: 2
Characters: 1472
Section: 1 Introduction
Path: ['1 Introduction']
Preview: ems with non-parametric memory which are trained from scratch for specific tasks, e.g.memory networks [64, 55], stack-augmentednetworks[25]andmemorylayers[30].Incontrast,weexploreasettingwhereboth par


In [48]:
def chunk_structural_unit(
    unit,
    chunk_size=1500,
    overlap=200
):
    text = unit.text.strip()

    if not text:
        return []

    if len(text) <= chunk_size:
        return [{
            "document": unit.document,
            "source_type": unit.source_type,
            "text": text,
            "metadata": {
                **unit.metadata,
                "section_title": unit.title,
                "section_level": unit.level,
                "chunk_index": 0,
            }
        }]

    chunks = []

    start = 0
    chunk_index = 0

    while start < len(text):

        end = min(start + chunk_size, len(text))

        # Back up to a whitespace boundary
        if end < len(text):
            boundary = text.rfind(" ", start, end)

            if boundary > start:
                end = boundary

        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "document": unit.document,
                "source_type": unit.source_type,
                "text": chunk_text,
                "metadata": {
                    **unit.metadata,
                    "section_title": unit.title,
                    "section_level": unit.level,
                    "chunk_index": chunk_index,
                }
            })

            chunk_index += 1

        if end >= len(text):
            break

        overlap_start = max(end - overlap, 0)

        if overlap_start > 0:
            boundary = text.find(" ", overlap_start, end)

        if boundary != -1:
            overlap_start = boundary + 1

    start = overlap_start

    return chunks

In [ ]:
intro_chunks = chunk_structural_unit(intro_unit)

for i, chunk in enumerate(intro_chunks):
    print("=" * 80)
    print("Chunk:", i)
    print("Characters:", len(chunk["text"]))
    print("Preview:", chunk["text"][:200])
    print("End:", chunk["text"][-100:])

In [ ]:
def chunk_dicts_to_objects(chunk_dicts):
    chunks = []

    for i, item in enumerate(chunk_dicts):
        chunks.append(
            Chunk(
                chunk_id=f"{item['document']}::chunk_{i}",
                document=item["document"],
                source_type=item["source_type"],
                text=item["text"],
                metadata=item["metadata"],
            )
        )

    return chunks

In [ ]:
intro_chunk_dicts = chunk_structural_unit(intro_unit)

intro_chunks = chunk_dicts_to_objects(intro_chunk_dicts)

print("Chunks:", len(intro_chunks))

for chunk in intro_chunks:
    print("-" * 80)
    print("ID:", chunk.chunk_id)
    print("Document:", chunk.document)
    print("Source:", chunk.source_type)
    print("Section:", chunk.metadata.get("section_title"))
    print("Path:", chunk.metadata.get("section_path"))
    print("Characters:", len(chunk.text))

In [ ]:
def chunk_dicts_to_objects(chunk_dicts, unit_index=0):
    chunks = []

    for chunk_index, item in enumerate(chunk_dicts):
        chunk_id = (
            f"{item['document']}"
            f"::unit_{unit_index}"
            f"::chunk_{chunk_index}"
        )

        chunks.append(
            Chunk(
                chunk_id=chunk_id,
                document=item["document"],
                source_type=item["source_type"],
                text=item["text"],
                metadata=item["metadata"],
            )
        )

    return chunks

In [ ]:
all_docx_units = []

for document in {
    d["document"] for d in docx_records
}:

    records = [
        d for d in docx_records
        if d["document"] == document
    ]

    sections = build_docx_sections(records)

    units = sections_to_structural_units(sections)

    all_docx_units.extend(units)

print("DOCX structural units:", len(all_docx_units))

In [ ]:
all_docx_chunks = []

for unit_index, unit in enumerate(all_docx_units):

    chunk_dicts = chunk_structural_unit(unit)

    chunks = chunk_dicts_to_objects(
        chunk_dicts,
        unit_index=unit_index
    )

    all_docx_chunks.extend(chunks)

print("DOCX chunks:", len(all_docx_chunks))

In [ ]:
for chunk in all_docx_chunks[:10]:

    print("=" * 80)
    print("ID:", chunk.chunk_id)
    print("Section:", chunk.metadata.get("section_title"))
    print("Level:", chunk.metadata.get("section_level"))
    print("Path:", chunk.metadata.get("section_path"))
    print("Characters:", len(chunk.text))
    print("Preview:", chunk.text[:150])

In [ ]:
# Build all DOCX structural units
all_docx_units = []

for document in {d["document"] for d in docx_records}:
    records = [
        d for d in docx_records
        if d["document"] == document
    ]

    sections = build_docx_sections(records)
    units = sections_to_structural_units(sections)

    all_docx_units.extend(units)


# Chunk all structural units
all_docx_chunks = []

for unit_index, unit in enumerate(all_docx_units):
    chunk_dicts = chunk_structural_unit(unit)

    chunks = chunk_dicts_to_objects(
        chunk_dicts,
        unit_index=unit_index
    )

    all_docx_chunks.extend(chunks)


print("DOCX records:", len(docx_records))
print("Structural units:", len(all_docx_units))
print("Final chunks:", len(all_docx_chunks))

In [ ]:
chunk_lengths = [
    len(chunk.text)
    for chunk in all_docx_chunks
]

print("Min:", min(chunk_lengths))
print("Max:", max(chunk_lengths))
print("Average:", sum(chunk_lengths) / len(chunk_lengths))

In [ ]:
oversized = [
    chunk for chunk in all_docx_chunks
    if len(chunk.text) > 1500
]

print("Oversized chunks:", len(oversized))

for chunk in oversized[:10]:
    print(
        len(chunk.text),
        chunk.metadata.get("section_title")
    )

In [ ]:
missing_provenance = [
    chunk for chunk in all_docx_chunks
    if not chunk.document
    or not chunk.source_type
    or "section_path" not in chunk.metadata
]

print("Missing provenance:", len(missing_provenance))

In [ ]:
pdf_records = [
    d for d in documents
    if d["source_type"] == "pdf"
]

print("PDF records:", len(pdf_records))

for record in pdf_records[:10]:
    print("=" * 80)
    print("Document:", record["document"])
    print("Location:", record["location"])
    print("Text:", record["text"][:200])
    print("Metadata:", record["metadata"])

In [ ]:
from collections import Counter

pdf_metadata_keys = Counter()

for record in pdf_records:
    pdf_metadata_keys.update(
        record.get("metadata", {}).keys()
    )

print("PDF metadata keys:")

for key, count in pdf_metadata_keys.most_common():
    print(f"{key}: {count}")

In [ ]:
location_patterns = Counter(
    record["location"]
    for record in pdf_records
)

print("PDF locations:")
for location, count in location_patterns.most_common(20):
    print(location, ":", count)
